In [2]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

from config import Config
from feature_engineering import drop_features, engineer_features
from feature_selection import run_feature_selection_pipeline
from plotting import plot_all

import pandas as pd
%matplotlib inline

In [5]:
df = pd.read_csv(Config.LMD_2022_PATH, low_memory=False)

useful_cols = []
for col in df.columns:
    if not col.startswith('Unnamed'):
        useful_cols.append(col)
df = df[useful_cols].copy()

df.rename(columns={'Label': 'isMalicious'}, inplace=True)

valid_rows = (df['isMalicious'] == '1') | (df['isMalicious'] == '0')
df = df[valid_rows].copy()
df['isMalicious'] = df['isMalicious'].astype(int)

print('Rows:', len(df))
print(df['isMalicious'].value_counts())

Rows: 293548
isMalicious
1    191194
0    102354
Name: count, dtype: int64


In [15]:
dataframe = df.copy()

METADATA_COLUMNS = [
    "SystemTime",
    "UtcTime",
    "CreationUtcTime",
    "PreviousCreationUtcTime",
    "User",
    "Name",
    "Version",
    "StartFunction",
]

GUID_COLUMNS = [
    "Guid",
    "ProcessGuid",
    "SourceProcessGUID",
    "TargetProcessGUID",
    "ParentProcessGuid",
    "SourceProcessGuid",
    "TargetProcessGuid",
    "LogonGuid",
]

ID_COLUMNS = [
    "ProcessID",
    "ProcessId",
    "ThreadID",
    "Execution_ProcessID",
    "EventRecordID",
    "SourceProcessId",
    "SourceThreadId",
    "TargetProcessId",
    "UserID",
]

LEAK_RISK_COLUMNS = [
    "CommandLine",
    "ParentCommandLine",
    "Hashes",
    "Hash",
    "CallTrace",
    "TargetFilename",
    "TargetObject",
    "Details",
    "Contents",
    "QueryResults",
    "ConfigurationFileHash",
]

REDUNDANT_COLUMNS = [
    "Task",
    "SignatureStatus",
    "DestinationIsIpv6",
]

columns_to_drop = list(dict.fromkeys(
    METADATA_COLUMNS +
    GUID_COLUMNS +
    ID_COLUMNS +
    LEAK_RISK_COLUMNS +
    REDUNDANT_COLUMNS
))

existing_columns_to_drop = [
    col for col in columns_to_drop
    if col in dataframe.columns
]

cleaned_dataframe = dataframe.drop(columns=existing_columns_to_drop)

print("Columns requested to drop:", len(columns_to_drop))
print("Columns found and dropped:", len(existing_columns_to_drop))
print("Original shape:", dataframe.shape)
print("Cleaned shape:", cleaned_dataframe.shape)

column_summary = pd.DataFrame({
    "column": cleaned_dataframe.columns,
    "distinct_value_count": [cleaned_dataframe[col].nunique(dropna=False) for col in cleaned_dataframe.columns],
    "missing_count": [cleaned_dataframe[col].isna().sum() for col in cleaned_dataframe.columns],
    "non_null_count": [cleaned_dataframe[col].notna().sum() for col in cleaned_dataframe.columns],
    "dtype": [str(cleaned_dataframe[col].dtype) for col in cleaned_dataframe.columns],
    "is_categorical": [
        pd.api.types.is_object_dtype(cleaned_dataframe[col]) or
        pd.api.types.is_string_dtype(cleaned_dataframe[col]) or
        pd.api.types.is_categorical_dtype(cleaned_dataframe[col]) or
        pd.api.types.is_bool_dtype(cleaned_dataframe[col]) or
        (
            pd.api.types.is_integer_dtype(cleaned_dataframe[col]) and
            cleaned_dataframe[col].nunique(dropna=False) <= 20
        )
        for col in cleaned_dataframe.columns
    ],
})

column_summary = column_summary.sort_values(
    by=["is_categorical", "distinct_value_count"],
    ascending=[False, False]
).reset_index(drop=True)

column_summary.to_csv("column_value_counts.csv", index=False)

column_summary

Columns requested to drop: 39
Columns found and dropped: 39
Original shape: (293548, 94)
Cleaned shape: (293548, 55)


/tmp/ipykernel_21964/1774806238.py:86: Pandas4Warning: is_categorical_dtype is deprecated and will be removed in a future version. Use isinstance(dtype, pd.CategoricalDtype) instead
  pd.api.types.is_categorical_dtype(cleaned_dataframe[col]) or


,column,distinct_value_count,missing_count,non_null_count,dtype,is_categorical
0,ParentProcessId,1060,0,293548,str,True
1,DestinationIp,598,0,293548,str,True
2,Image,512,0,293548,str,True
3,OriginalFileName,471,0,293548,str,True
4,Description,422,0,293548,str,True
5,QueryName,398,0,293548,str,True
6,NewThreadId,388,88,293460,str,True
7,IsExecutable,341,4386,289162,str,True
8,Archived,330,302,293246,str,True
9,ImageLoaded,297,0,293548,str,True
